# CUDA Kernel 面试主线 · 第 6/12 课：稳定 Softmax 与朴素复杂度

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：写出减最大值的稳定 softmax，并准确分析朴素 kernel 的 O(N²) 行扫描。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：Softmax 把 logits 变成正且和为 1 的分布；指数使它对溢出和下溢敏感。

## 核心心智模型

### 1. 它是什么，解决什么问题

Softmax 把 logits 变成正且和为 1 的分布；指数使它对溢出和下溢敏感。

### 2. 它如何工作

先求行最大值 m，再算 sum exp(x-m)，最后输出 exp(x-m)/sum；平移不改变结果。

### 3. 正确性条件与常见误区

全被 mask 的行需要定义行为；分母必须对应同一组有效元素，不能把 padding 纳入统计。

### 4. 性能与工程取舍

一线程一输出会让每行 max/sum 重算 N 次，教学清晰但访存和工作量 O(N²)。

## 具体演示

logits [1000,1001] 直接 exp 溢出；减 1001 后变为 [-1,0]，稳定且比例不变。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐最终稳定输出。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/06_softmax_naive.cu
#include <float.h>
#include <cuda_runtime.h>

// Naive Softmax: 对 [M, N] 的每一行做 softmax。
//
// 这个版本保留“一个线程负责一个输出元素”的写法，
// 适合面试里先讲清楚 softmax 的数学公式和它为什么低效。
//
// output[row, col] = exp(input[row, col] - max(row)) / sum(exp(input[row, i) - max(row)))
//
// 低效点：
// 对同一行来说，max(row) 和 sum(row) 是所有 col 共享的，
// 但这里每个输出元素线程都会重新扫完整行求 max 和 sum。
// 所以每行复杂度接近 O(N^2)，优化版应改成一个 block 合作处理一行。
__global__ void softmax_naive_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int M,
    int N
) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < M && col < N) {
        float max_val = -FLT_MAX;

        // 第一次扫整行，求最大值，避免 exp 溢出。
        for (int i = 0; i < N; ++i) {
            max_val = fmaxf(max_val, input[row * N + i]);
        }

        float sum_exp = 0.0f;

        // 第二次扫整行，求 denominator。
        for (int i = 0; i < N; ++i) {
            sum_exp += expf(input[row * N + i] - max_val);
        }

        // 当前线程只写自己负责的一个 output[row, col]。
        output[row * N + col] = ______;  // TODO: 稳定归一化
    }
}

void launch_softmax_naive(
    const float* input,
    float* output,
    int M,
    int N,
    cudaStream_t stream
) {
    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x, (M + block.y - 1) / block.y);
    softmax_naive_kernel<<<grid, block, 0, stream>>>(input, output, M, N);
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/06_softmax_naive.cu -o /tmp/06_softmax_naive.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“稳定 Softmax 与朴素复杂度”的工作机制。

**你的答案：**


### Q2

只在分母减 max、分子不减会怎样？

**你的答案：**


### Q3

为什么这个 naive kernel 在 N 增大时比“一 block 一行”恶化得更快？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
%%writefile /tmp/06_softmax_naive.cu
#include <float.h>
#include <cuda_runtime.h>

// Naive Softmax: 对 [M, N] 的每一行做 softmax。
//
// 这个版本保留“一个线程负责一个输出元素”的写法，
// 适合面试里先讲清楚 softmax 的数学公式和它为什么低效。
//
// output[row, col] = exp(input[row, col] - max(row)) / sum(exp(input[row, i) - max(row)))
//
// 低效点：
// 对同一行来说，max(row) 和 sum(row) 是所有 col 共享的，
// 但这里每个输出元素线程都会重新扫完整行求 max 和 sum。
// 所以每行复杂度接近 O(N^2)，优化版应改成一个 block 合作处理一行。
__global__ void softmax_naive_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int M,
    int N
) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < M && col < N) {
        float max_val = -FLT_MAX;

        // 第一次扫整行，求最大值，避免 exp 溢出。
        for (int i = 0; i < N; ++i) {
            max_val = fmaxf(max_val, input[row * N + i]);
        }

        float sum_exp = 0.0f;

        // 第二次扫整行，求 denominator。
        for (int i = 0; i < N; ++i) {
            sum_exp += expf(input[row * N + i] - max_val);
        }

        // 当前线程只写自己负责的一个 output[row, col]。
        output[row * N + col] = expf(input[row * N + col] - max_val) / sum_exp;
    }
}

void launch_softmax_naive(
    const float* input,
    float* output,
    int M,
    int N,
    cudaStream_t stream
) {
    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x, (M + block.y - 1) / block.y);
    softmax_naive_kernel<<<grid, block, 0, stream>>>(input, output, M, N);
}


### Q1 参考答案

先求行最大值 m，再算 sum exp(x-m)，最后输出 exp(x-m)/sum；平移不改变结果。

### Q2 参考答案

判断时先检查本课不变量：全被 mask 的行需要定义行为；分母必须对应同一组有效元素，不能把 padding 纳入统计。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：一线程一输出会让每行 max/sum 重算 N 次，教学清晰但访存和工作量 O(N²)。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。